# Cohesion-Sensitive Power Indices: Comprehensive Analysis & Figure Generation

This notebook implements the cohesion-sensitive power indices methodology and generates all figures for the paper. It includes:
- Core computational framework for cohesion-Shapley indices
- Analysis of an apex game example
- Application to German Bundestag 2025
- Application to French Assemblée Nationale 2024 (bloc and party models)

All computations use normalized cohesion-Shapley indices with ideology-based and cordon sanitaire cohesion functions.

In [1]:
import matplotlib
matplotlib.use('Agg')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations
import math
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

In [2]:
def shapley_size_weight(k, n):
    """
    Compute Shapley size weight: k!(n-k-1)!/n!
    
    Args:
        k: size of coalition (integer)
        n: total number of players (integer)
    
    Returns:
        Shapley size weight (float)
    """
    if k < 0 or k >= n:
        return 0.0
    
    # k! * (n-k-1)! / n!
    numerator = math.factorial(k) * math.factorial(n - k - 1)
    denominator = math.factorial(n)
    return numerator / denominator


def cohesion_shapley(players, weights, threshold, kappa_func, b_val):
    """
    Compute normalized cohesion-Shapley index.
    
    The index for player i is computed as:
    φ_i = (1/n) * Σ_S⊆N\{i} w(S) * κ(S∪{i})^b * [v(S∪{i}) - v(S)]
    
    where:
    - w(S) = |S|!(n-|S|-1)!/n! is the Shapley size weight
    - κ(S∪{i}) is the cohesion value of the coalition with player i
    - [v(S∪{i}) - v(S)] is 1 if S∪{i} is winning and S is not (pivotality), 0 otherwise
    - b is the cohesion parameter
    
    Args:
        players: list of player identifiers
        weights: dict mapping player → seat count
        threshold: winning coalition threshold
        kappa_func: function(coalition_set) → cohesion value [0, 1]
        b_val: cohesion parameter (float ≥ 0)
    
    Returns:
        dict mapping player → normalized index value
    """
    n = len(players)
    indices = {p: 0.0 for p in players}
    
    # Enumerate all non-empty subsets of N\{i} for each player i
    for i, player_i in enumerate(players):
        other_players = [p for p in players if p != player_i]
        
        # Enumerate all subsets of N\{i} (including empty set)
        for r in range(len(other_players) + 1):
            for subset in combinations(other_players, r):
                coalition_S = set(subset)
                coalition_S_union_i = coalition_S | {player_i}
                
                # Compute Shapley size weight
                k = len(coalition_S)
                w_S = shapley_size_weight(k, n)
                
                # Compute cohesion value
                kappa_S_union_i = kappa_func(coalition_S_union_i)
                
                # Check pivotality
                weight_S = sum(weights[p] for p in coalition_S)
                weight_S_union_i = weight_S + weights[player_i]
                
                is_S_winning = weight_S >= threshold
                is_S_union_i_winning = weight_S_union_i >= threshold
                
                pivotal = (not is_S_winning) and is_S_union_i_winning
                
                if pivotal:
                    contribution = w_S * (kappa_S_union_i ** b_val)
                    indices[player_i] += contribution
    
    # Normalize by dividing by n
    for p in players:
        indices[p] /= n
    
    return indices


print("Core functions loaded successfully.")

Core functions loaded successfully.


## 1. Apex Game (Figure 1)

In [3]:
# Apex game setup
apex_players = ['A', 'B', 'C']
apex_weights = {'A': 45, 'B': 35, 'C': 20}
apex_threshold = 51

# Hardcoded cohesion values
apex_cohesion_values = {
    frozenset(['A', 'B']): 0.20,
    frozenset(['A', 'C']): 0.05,
    frozenset(['B', 'C']): 0.90,
    frozenset(['A', 'B', 'C']): 0.05
}

def apex_kappa(coalition):
    """Cohesion function for apex game."""
    key = frozenset(coalition)
    if key in apex_cohesion_values:
        return apex_cohesion_values[key]
    return 1.0  # Single player always has κ=1

# Compute indices for range of b values
b_values_apex = np.linspace(0.001, 4.0, 300)
apex_results = {player: [] for player in apex_players}

for b in b_values_apex:
    indices = cohesion_shapley(apex_players, apex_weights, apex_threshold, apex_kappa, b)
    for player in apex_players:
        apex_results[player].append(indices[player])

# Plot Figure 1: Apex Game
fig, ax = plt.subplots(figsize=(8, 5.5))

colors = {'A': '#2166ac', 'B': '#b2182b', 'C': '#4d9221'}
linewidth = 1.4

for player in apex_players:
    ax.plot(b_values_apex, apex_results[player], 
            label=f'Player {player}', 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('Apex Game: Power Index Variation', fontsize=12, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black')
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

fig.tight_layout()
fig.savefig('fig1_apex.pdf', dpi=300, bbox_inches='tight')
fig.savefig('fig1_apex.png', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 1 (Apex Game) generated successfully.")
print(f"At b=1: A={apex_results['A'][np.argmin(np.abs(b_values_apex - 1.0))]:.4f}, "
      f"B={apex_results['B'][np.argmin(np.abs(b_values_apex - 1.0))]:.4f}, "
      f"C={apex_results['C'][np.argmin(np.abs(b_values_apex - 1.0))]:.4f}")

Figure 1 (Apex Game) generated successfully.
At b=1: A=0.0138, B=0.0610, C=0.0527


## 2. Germany: 21st Bundestag 2025 (Figure 2)

In [4]:
# Load Bundestag 2025 data
bt_df = pd.read_csv('../data/bundestag_2025.csv')

# Construct analysis groups
# CDU/CSU combined: 164 + 44 = 208 seats
# Weighted CHES: (5.86*164 + 7.19*44)/208 = 6.14
bt_groups = {
    'CDU/CSU': {'seats': 208, 'ches': 6.14},
    'AfD': {'seats': 77, 'ches': 9.25},
    'SPD': {'seats': 112, 'ches': 3.26},
    'Grüne': {'seats': 118, 'ches': 2.41},
    'Linke': {'seats': 24, 'ches': 1.84}
}

bt_players = list(bt_groups.keys())
bt_weights = {p: bt_groups[p]['seats'] for p in bt_players}
bt_ches = {p: bt_groups[p]['ches'] for p in bt_players}
bt_threshold = 316  # > 315 (half of 630)

# Scenario A: Pure ideology (based on CHES distance)
def bt_kappa_ideology(coalition):
    """Ideology-based cohesion: κ(S) = 1 / (1 + max|λ_i - λ_j|)."""
    if len(coalition) <= 1:
        return 1.0
    
    ches_values = [bt_ches[p] for p in coalition]
    max_distance = max(ches_values) - min(ches_values)
    return 1.0 / (1.0 + max_distance)

# Scenario B: Cordon sanitaire (AfD excluded)
def bt_kappa_cordon(coalition):
    """Cordon sanitaire: κ(S) = 0 if AfD ∈ S, else ideology-based."""
    if 'AfD' in coalition:
        return 0.0
    return bt_kappa_ideology(coalition)

# Compute indices for Scenario A and B
b_values_bt = np.linspace(0.001, 3.0, 200)
bt_results_A = {player: [] for player in bt_players}
bt_results_B = {player: [] for player in bt_players}

for b in b_values_bt:
    indices_A = cohesion_shapley(bt_players, bt_weights, bt_threshold, bt_kappa_ideology, b)
    indices_B = cohesion_shapley(bt_players, bt_weights, bt_threshold, bt_kappa_cordon, b)
    
    for player in bt_players:
        bt_results_A[player].append(indices_A[player])
        bt_results_B[player].append(indices_B[player])

# Plot Figure 2: Bundestag 2025
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

colors = {
    'CDU/CSU': '#1a1a1a',
    'AfD': '#2166ac',
    'SPD': '#b2182b',
    'Grüne': '#4d9221',
    'Linke': '#7b3294'
}

linewidth = 1.4

# Panel A: Ideology-based
ax = axes[0]
for player in bt_players:
    ax.plot(b_values_bt, bt_results_A[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(A) Ideology-based cohesion', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=9)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

# Panel B: Cordon sanitaire
ax = axes[1]
for player in bt_players:
    ax.plot(b_values_bt, bt_results_B[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(B) Cordon sanitaire (AfD excluded)', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=9)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

fig.tight_layout()
fig.savefig('fig2_bundestag.pdf', dpi=300, bbox_inches='tight')
fig.savefig('fig2_bundestag.png', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 2 (Bundestag 2025) generated successfully.")

# Print summary at b=1
idx_b1 = np.argmin(np.abs(b_values_bt - 1.0))
print("\nBundestag at b=1:")
print("Scenario A (Ideology):")
for player in bt_players:
    print(f"  {player}: {bt_results_A[player][idx_b1]:.4f}")
print("Scenario B (Cordon):")
for player in bt_players:
    print(f"  {player}: {bt_results_B[player][idx_b1]:.4f}")

Figure 2 (Bundestag 2025) generated successfully.

Bundestag at b=1:
Scenario A (Ideology):
  CDU/CSU: 0.0161
  AfD: 0.0012
  SPD: 0.0072
  Grüne: 0.0067
  Linke: 0.0012
Scenario B (Cordon):
  CDU/CSU: 0.0106
  AfD: 0.0000
  SPD: 0.0039
  Grüne: 0.0034
  Linke: 0.0000


## 3. France: 17th Assemblée Nationale 2024

In [5]:
# Load France 2024 data
fr_df = pd.read_csv('../data/france_2024.csv')

# BLOC MODEL (5 groups)
# NFP: LFI(71) + PS(69) + Écologistes(38) + GDR(17) = 195
# Weighted CHES: (1.25*71 + 3.00*69 + 2.50*38 + 1.13*17)/195 = 2.19

# Ensemble: Renaissance(92) + MoDem(36) + Horizons(34) = 162
# Weighted CHES: (6.33*92 + 6.13*36 + 6.20*34)/162 = 6.24

# LR: 49 seats, CHES 7.88

# RN+UDR: RN(122) + UDR(17) = 139
# Weighted CHES: (9.75*122 + 8.50*17)/139 = 9.60

# Others: LIOT(22) + non-inscrits(10) = 32, CHES ≈ 5.00

fr_bloc_groups = {
    'NFP': {'seats': 195, 'ches': 2.19},
    'Ensemble': {'seats': 162, 'ches': 6.24},
    'LR': {'seats': 49, 'ches': 7.88},
    'RN': {'seats': 139, 'ches': 9.60},
    'Others': {'seats': 32, 'ches': 5.00}
}

# PARTY MODEL (6 groups)
# LFI: 71 seats, CHES 1.25
# PS+Écologistes+GDR: 69+38+17 = 124
# Weighted CHES: (3.00*69 + 2.50*38 + 1.13*17)/124 = 2.55
# Ensemble: 162 seats, CHES 6.24 (same)
# LR: 49 seats, CHES 7.88 (same)
# RN: 139 seats, CHES 9.60 (same)
# Others: 32 seats, CHES 5.00 (same)

fr_party_groups = {
    'LFI': {'seats': 71, 'ches': 1.25},
    'PS-Verts': {'seats': 124, 'ches': 2.55},
    'Ensemble': {'seats': 162, 'ches': 6.24},
    'LR': {'seats': 49, 'ches': 7.88},
    'RN': {'seats': 139, 'ches': 9.60},
    'Others': {'seats': 32, 'ches': 5.00}
}

fr_threshold = 289  # > 288 (half of 577)

# Ideology-based cohesion function (generic)
def fr_kappa_ideology(coalition, ches_dict):
    """Ideology-based cohesion: κ(S) = 1 / (1 + max|λ_i - λ_j|)."""
    if len(coalition) <= 1:
        return 1.0
    
    ches_values = [ches_dict[p] for p in coalition]
    max_distance = max(ches_values) - min(ches_values)
    return 1.0 / (1.0 + max_distance)

print("France data loaded and prepared for bloc and party models.")

France data loaded and prepared for bloc and party models.


In [6]:
# BLOC MODEL ANALYSIS
fr_bloc_players = list(fr_bloc_groups.keys())
fr_bloc_weights = {p: fr_bloc_groups[p]['seats'] for p in fr_bloc_players}
fr_bloc_ches = {p: fr_bloc_groups[p]['ches'] for p in fr_bloc_players}

# Scenario A: Pure ideology
def fr_bloc_kappa_A(coalition):
    return fr_kappa_ideology(coalition, fr_bloc_ches)

# Scenario B: Cordon sanitaire vs RN
def fr_bloc_kappa_B(coalition):
    if 'RN' in coalition:
        return 0.0
    return fr_bloc_kappa_A(coalition)

# Scenario C: Double cordon vs RN and NFP
def fr_bloc_kappa_C(coalition):
    if 'RN' in coalition or 'NFP' in coalition:
        return 0.0
    return fr_bloc_kappa_A(coalition)

# Compute indices for all scenarios
b_values_fr = np.linspace(0.001, 3.0, 200)
fr_bloc_results_A = {player: [] for player in fr_bloc_players}
fr_bloc_results_B = {player: [] for player in fr_bloc_players}
fr_bloc_results_C = {player: [] for player in fr_bloc_players}

for b in b_values_fr:
    indices_A = cohesion_shapley(fr_bloc_players, fr_bloc_weights, fr_threshold, fr_bloc_kappa_A, b)
    indices_B = cohesion_shapley(fr_bloc_players, fr_bloc_weights, fr_threshold, fr_bloc_kappa_B, b)
    indices_C = cohesion_shapley(fr_bloc_players, fr_bloc_weights, fr_threshold, fr_bloc_kappa_C, b)
    
    for player in fr_bloc_players:
        fr_bloc_results_A[player].append(indices_A[player])
        fr_bloc_results_B[player].append(indices_B[player])
        fr_bloc_results_C[player].append(indices_C[player])

print("Bloc model computation complete.")

Bloc model computation complete.


In [7]:
# Plot Figure 3: France Bloc Model
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8))

colors = {
    'NFP': '#b2182b',
    'Ensemble': '#f4a460',
    'LR': '#2166ac',
    'RN': '#1a1a1a',
    'Others': '#999999'
}

linewidth = 1.4

# Panel A: Pure ideology
ax = axes[0]
for player in fr_bloc_players:
    ax.plot(b_values_fr, fr_bloc_results_A[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(A) Ideology-based', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=9)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

# Panel B: Cordon vs RN
ax = axes[1]
for player in fr_bloc_players:
    ax.plot(b_values_fr, fr_bloc_results_B[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(B) Cordon sanitaire (RN excluded)', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=9)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

# Panel C: Double cordon
ax = axes[2]
for player in fr_bloc_players:
    ax.plot(b_values_fr, fr_bloc_results_C[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(C) Double cordon (RN, NFP excluded)', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=9)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

fig.tight_layout()
fig.savefig('fig3_france_bloc.pdf', dpi=300, bbox_inches='tight')
fig.savefig('fig3_france_bloc.png', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 3 (France Bloc Model) generated successfully.")

# Print summary at b=1
idx_b1 = np.argmin(np.abs(b_values_fr - 1.0))
print("\nFrance Bloc at b=1:")
print("Scenario A (Ideology):")
for player in fr_bloc_players:
    print(f"  {player}: {fr_bloc_results_A[player][idx_b1]:.4f}")
print("Scenario B (Cordon RN):")
for player in fr_bloc_players:
    print(f"  {player}: {fr_bloc_results_B[player][idx_b1]:.4f}")
print("Scenario C (Double cordon):")
for player in fr_bloc_players:
    print(f"  {player}: {fr_bloc_results_C[player][idx_b1]:.4f}")

Figure 3 (France Bloc Model) generated successfully.

France Bloc at b=1:
Scenario A (Ideology):
  NFP: 0.0098
  Ensemble: 0.0127
  LR: 0.0000
  RN: 0.0108
  Others: 0.0000
Scenario B (Cordon RN):
  NFP: 0.0058
  Ensemble: 0.0058
  LR: 0.0000
  RN: 0.0000
  Others: 0.0000
Scenario C (Double cordon):
  NFP: 0.0000
  Ensemble: 0.0000
  LR: 0.0000
  RN: 0.0000
  Others: 0.0000


In [8]:
# PARTY MODEL ANALYSIS
fr_party_players = list(fr_party_groups.keys())
fr_party_weights = {p: fr_party_groups[p]['seats'] for p in fr_party_players}
fr_party_ches = {p: fr_party_groups[p]['ches'] for p in fr_party_players}

# Scenario A: Pure ideology
def fr_party_kappa_A(coalition):
    return fr_kappa_ideology(coalition, fr_party_ches)

# Scenario B: Cordon sanitaire vs RN
def fr_party_kappa_B(coalition):
    if 'RN' in coalition:
        return 0.0
    return fr_party_kappa_A(coalition)

# Scenario C: Double cordon vs RN and LFI
def fr_party_kappa_C(coalition):
    if 'RN' in coalition or 'LFI' in coalition:
        return 0.0
    return fr_party_kappa_A(coalition)

# Compute indices for all scenarios
fr_party_results_A = {player: [] for player in fr_party_players}
fr_party_results_B = {player: [] for player in fr_party_players}
fr_party_results_C = {player: [] for player in fr_party_players}

for b in b_values_fr:
    indices_A = cohesion_shapley(fr_party_players, fr_party_weights, fr_threshold, fr_party_kappa_A, b)
    indices_B = cohesion_shapley(fr_party_players, fr_party_weights, fr_threshold, fr_party_kappa_B, b)
    indices_C = cohesion_shapley(fr_party_players, fr_party_weights, fr_threshold, fr_party_kappa_C, b)
    
    for player in fr_party_players:
        fr_party_results_A[player].append(indices_A[player])
        fr_party_results_B[player].append(indices_B[player])
        fr_party_results_C[player].append(indices_C[player])

print("Party model computation complete.")

Party model computation complete.


In [9]:
# Plot Figure 4: France Party Model
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8))

colors = {
    'LFI': '#e63946',
    'PS-Verts': '#f4a460',
    'Ensemble': '#457b9d',
    'LR': '#2166ac',
    'RN': '#1a1a1a',
    'Others': '#999999'
}

linewidth = 1.4

# Panel A: Pure ideology
ax = axes[0]
for player in fr_party_players:
    ax.plot(b_values_fr, fr_party_results_A[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(A) Ideology-based', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

# Panel B: Cordon vs RN
ax = axes[1]
for player in fr_party_players:
    ax.plot(b_values_fr, fr_party_results_B[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(B) Cordon sanitaire (RN excluded)', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

# Panel C: Double cordon
ax = axes[2]
for player in fr_party_players:
    ax.plot(b_values_fr, fr_party_results_C[player], 
            label=player, 
            color=colors[player], 
            linewidth=linewidth)

ax.set_xlabel(r'Cohesion parameter $b$', fontsize=11)
ax.set_ylabel(r'Normalized cohesion-Shapley index', fontsize=11)
ax.set_title('(C) Double cordon (RN, LFI excluded)', fontsize=11, fontweight='bold')
ax.legend(loc='best', frameon=True, fancybox=False, edgecolor='black', fontsize=8)
ax.grid(True, alpha=0.3, linestyle=':')
ax.set_ylim([0, None])

fig.tight_layout()
fig.savefig('fig4_france_party.pdf', dpi=300, bbox_inches='tight')
fig.savefig('fig4_france_party.png', dpi=300, bbox_inches='tight')
plt.close()

print("Figure 4 (France Party Model) generated successfully.")

# Print summary at b=1
idx_b1 = np.argmin(np.abs(b_values_fr - 1.0))
print("\nFrance Party at b=1:")
print("Scenario A (Ideology):")
for player in fr_party_players:
    print(f"  {player}: {fr_party_results_A[player][idx_b1]:.4f}")
print("Scenario B (Cordon RN):")
for player in fr_party_players:
    print(f"  {player}: {fr_party_results_B[player][idx_b1]:.4f}")
print("Scenario C (Double cordon):")
for player in fr_party_players:
    print(f"  {player}: {fr_party_results_C[player][idx_b1]:.4f}")

Figure 4 (France Party Model) generated successfully.

France Party at b=1:
Scenario A (Ideology):
  LFI: 0.0014
  PS-Verts: 0.0047
  Ensemble: 0.0081
  LR: 0.0015
  RN: 0.0070
  Others: 0.0016
Scenario B (Cordon RN):
  LFI: 0.0008
  PS-Verts: 0.0028
  Ensemble: 0.0039
  LR: 0.0008
  RN: 0.0000
  Others: 0.0010
Scenario C (Double cordon):
  LFI: 0.0000
  PS-Verts: 0.0015
  Ensemble: 0.0015
  LR: 0.0004
  RN: 0.0000
  Others: 0.0006


## Summary of Key Findings

This notebook has computed normalized cohesion-Shapley power indices across multiple applications:

### 1. Apex Game
A theoretical example with three players demonstrating how the cohesion parameter b affects power distribution when coalition formation patterns are constrained by cohesion values.

### 2. German Bundestag 2025
Analysis of two scenarios: (A) pure ideology-based cohesion and (B) cordon sanitaire excluding AfD. The results show how exclusionary coalition constraints significantly redistribute power among mainstream parties.

### 3. French Assemblée Nationale 2024
Two structural models (bloc and party) with three scenarios each:
- **Bloc model:** NFP, Ensemble, LR, RN, Others
- **Party model:** LFI, PS-Verts, Ensemble, LR, RN, Others

Each model explores: (A) pure ideology, (B) cordon vs RN, and (C) double cordon (vs RN and LFI/NFP).

All figures have been saved as high-resolution PDFs and PNGs suitable for publication.